In [21]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

Useful functions

In [22]:
# Function to change the data type of a column to float

def convert_to_float(df):
    for col in df.columns:
        if df[col].dtype == 'object':
            try:
                df[col] = df[col].astype(float)
            except ValueError:
                pass  # Ignore non-convertible columns
    return df

# Function to compute the mode of a series
def mode(series):
    return series.mode()[0] if not series.mode().empty else np.nan

# Function to impute missing values using the mean or mode
def custom_imputer(df, col_name, option):
    if option == 'mean':
      mean = df.groupby('Region')[col_name].transform('mean')
      df[col_name] = df[col_name].fillna(mean)
      mean = df.groupby('Wilaya')[col_name].transform('mean')
      df[col_name] = df[col_name].fillna(mean)
      mean = mode(df[col_name])
      df[col_name] = df[col_name].fillna(mean)
    else :
      modes = df.groupby('Region')[col_name].transform(lambda x: mode(x))
      df[col_name] = df[col_name].fillna(modes)
      modes = df.groupby('Wilaya')[col_name].transform(lambda x: mode(x))
      df[col_name] = df[col_name].fillna(modes)
      global_mode = mode(df[col_name])
      df[col_name] = df[col_name].fillna(global_mode)

    return df

# Function to impute missing values using the KNN imputer
def knn_impute(df, col_name):
    # Initialize the KNN imputer
    knn_imputer = KNNImputer(n_neighbors=5)

    # Apply KNN imputation
    df[col_name] = knn_imputer.fit_transform(df[[col_name]])
    return df

# Function to adjust the imputed values to sum to 100
def adjust_values(row):
    total = row.sum()
    if total != 100:
        row = (row / total) * 100
    return row


Load and showcase our dataset


In [3]:
dataframe = pd.read_csv("./datasets/raw_dataset.csv")

In [4]:
print(dataframe['FAOSOIL'].unique())
print(dataframe['GLG'].unique())
print(dataframe['DOMSOI'].unique())

['Bc6-2ab' 'I-Bk-bc' 'Jc16-2/3a' 'Yk14-2a' 'Yk3-2a' 'Yk11-1/2ab' nan
 'Xh12-2a' 'Bk14-2c' 'Xk5-1a' 'Zt2-3a' 'La30-2bc' 'Bk2-bc' 'Bk14-2bc'
 'Yy4-1a' 'I-Rc-1/2bc' 'Yk3-2ab' 'Jc15-2/3a' 'Xk12-2a' 'Bk2-2ab' 'Lo38-bc'
 'Vc24-3a' 'Vp8-3a' 'Vc23-3a']
['Qe' 'K' 'pCm' 'Q' nan 'Kl' 'J' 'SEA' 'T']
['Bc' 'I' 'Jc' 'Yk' nan 'Xh' 'Bk' 'Xk' 'Zt' 'La' 'Yy' 'Lo' 'Vc' 'Vp']


In [9]:
print(dataframe.isna().sum().max())
dataframe = dataframe.astype(str)
dataframe.replace(",", ".", regex=True, inplace=True)
dataframe = convert_to_float(dataframe)

0


Encoding

In [5]:
# Encode 'GLG' column
dataframe['GLG'] = dataframe['GLG'].replace({'Qe': 0, 'K': 1, 'pCm': 2, 'Q': 3, 'Kl': 4, 'J': 5, 'SEA': 6, 'T': 7})

# Encode 'FAOSOIL' column
dataframe['FAOSOIL'] = dataframe['FAOSOIL'].replace({
    'Bc6-2ab': 0, 'I-Bk-bc': 1, 'Jc16-2/3a': 2, 'Yk14-2a': 3, 'Yk3-2a': 4, 'Yk11-1/2ab': 5,
    'Xh12-2a': 6, 'Bk14-2c': 7, 'Xk5-1a': 8, 'Zt2-3a': 9, 'La30-2bc': 10,
    'Bk2-bc': 11, 'Bk14-2bc': 12, 'Yy4-1a': 13, 'I-Rc-1/2bc': 14, 'Yk3-2ab': 15,
    'Jc15-2/3a': 16, 'Xk12-2a': 17, 'Bk2-2ab': 18, 'Lo38-bc': 19, 'Vc24-3a': 20,
    'Vp8-3a': 21, 'Vc23-3a': 22
})

# Encode 'DOMSOI' column
dataframe['DOMSOI'] = dataframe['DOMSOI'].replace({
    'Bc': 0, 'I': 1, 'Jc': 2, 'Yk': 3, 'Bk': 4, 'Xh': 5, 'Xk': 6, 'Zt': 7,
    'La': 8, 'Yy': 9, 'Lo': 10, 'Vc': 11, 'Vp': 12, 'X': 13, 'Lc': 14
})

C:\Users\zakmins\AppData\Local\Temp\ipykernel_9656\333116200.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataframe['GLG'] = dataframe['GLG'].replace({'Qe': 0, 'K': 1, 'pCm': 2, 'Q': 3, 'Kl': 4, 'J': 5, 'SEA': 6, 'T': 7})
C:\Users\zakmins\AppData\Local\Temp\ipykernel_9656\333116200.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataframe['FAOSOIL'] = dataframe['FAOSOIL'].replace({
C:\Users\zakmins\AppData\Local\Temp\ipykernel_9656\333116200.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will 

Get rid of NaN coordinates and handling missing values

In [6]:
dataframe = dataframe.dropna(subset=['X', 'Y'], how='all')

In [12]:
# Columns to impute
nan_impute  = {
    'mean' : ['pH', 'AREA'],
    'mode' : ['GLG', 'PERIMETER', 'GEO2_7G_', 'GEO2_7G_ID']
}

In [13]:
for option, col_names in nan_impute.items():
    for col_name in col_names:
      dataframe = custom_imputer(dataframe, col_name, option)

In [18]:
# Use KNN imputer to fill in the missing values (A%, L%, S%) Soil texture
knn_imputer = KNNImputer(n_neighbors=3)
dataframe[['A%', 'S%', 'L%']] = knn_imputer.fit_transform(dataframe[['A%', 'S%', 'L%']])

# Apply a function to each row to make sure the sum of A%, L%, and S% is 100%
dataframe[["A%", "L%", "S%"]] = dataframe[["A%", "L%", "S%"]].apply(adjust_values, axis=1)

Handling Outliers

In [16]:
#Outliers
dataframe['pH'].replace("0.817", "8.17", regex=True, inplace=True)

#handle outliers for CE and CEC
dataframe.loc[dataframe['CE'] > 30, 'CE'] = np.nan
dataframe.loc[dataframe['CEC'] > 100, 'CEC'] = np.nan

C:\Users\zakmins\AppData\Local\Temp\ipykernel_9656\1725191797.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dataframe['pH'].replace("0.817", "8.17", regex=True, inplace=True)


In [19]:
cols_to_impute_knn = ['OM', 'CE', 'CEC']

for col_name in cols_to_impute_knn:
    dataframe = knn_impute(dataframe, col_name)

In [20]:
dataframe.to_csv('./datasets/preprocessed_dataset.csv', index=False)